# NASA FIRMS Thermal Hotspot Data - Exploratory Data Analysis (EDA)

**Project**: AI-Based Detection and Classification of Industrial Fires and Persistent Thermal Sources Using NASA FIRMS, OSM and Satellite Data (SIH 2026)

### Goals of this Notebook:
1. **Load Raw & Cleaned FIRMS Datasets** using the modular pipeline.
2. **Inspect Data Hygiene**: Missing values, duplicate records, data types.
3. **Geospatial & Temporal Profiling**: Coordinate bounds (Latitude/Longitude) and Date-Time ranges.
4. **Physical Attribute Distributions**:
   - Fire Radiative Power (FRP in MW)
   - Brightness Temperatures (Kelvin)
   - Detection Confidence (MODIS % / VIIRS categorical)

In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.firms_loader import load_firms_data
from src.data.firms_validator import validate_firms_schema, validate_coordinates, validate_physical_values
from src.data.firms_cleaner import clean_firms_data
from src.data.firms_eda import generate_eda_summary, print_eda_report
from configs.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, RAW_FIRMS_DEFAULT, PROCESSED_FIRMS_DEFAULT

# Set aesthetic styling for charts
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

## 1. Pipeline Execution & Data Ingestion

In [ ]:
# Load cleaned data if available, or load raw data through the pipeline
if PROCESSED_FIRMS_DEFAULT.exists():
    df = pd.read_csv(PROCESSED_FIRMS_DEFAULT, parse_dates=["acq_datetime"])
    print(f"Loaded processed FIRMS dataset: {len(df):,} records.")
elif RAW_FIRMS_DEFAULT.exists():
    print(f"Processing raw dataset at {RAW_FIRMS_DEFAULT}...")
    raw_df = load_firms_data(RAW_FIRMS_DEFAULT)
    df, stats = clean_firms_data(raw_df, output_path=PROCESSED_FIRMS_DEFAULT)
    print(f"Cleaned dataset created: {len(df):,} records.")
else:
    print("No CSV file found in data/raw/. Please place your downloaded FIRMS CSV file in data/raw/firms_raw.csv")
    df = pd.DataFrame()  # Empty fallback

## 2. Dataset Overview & Schema Inspection

In [ ]:
if not df.empty:
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
    print("Data Types & Non-Null Counts:")
    print(df.info())
    display(df.head(10))

## 3. Automated EDA Summary Report

In [ ]:
if not df.empty:
    eda_summary = generate_eda_summary(df, report_title="FIRMS Hotspot Dataset Overview")
    _ = print_eda_report(eda_summary)

## 4. Missing Values & Duplicate Analysis

In [ ]:
if not df.empty:
    missing = df.isna().sum()
    duplicates_count = df.duplicated().sum()
    print(f"Total exact duplicate records: {duplicates_count}")
    
    plt.figure(figsize=(10, 4))
    missing.plot(kind="bar", color="#e74c3c", edgecolor="black")
    plt.title("Missing Values Count per Column", fontsize=14, fontweight="bold")
    plt.ylabel("Null Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 5. Temporal Distribution (Acquisitions Over Time)

In [ ]:
if not df.empty and "acq_date" in df.columns:
    plt.figure(figsize=(12, 4))
    df.groupby("acq_date").size().plot(kind="line", marker="o", color="#2980b9", linewidth=2)
    plt.title("Hotspot Detections Over Time (Daily Count)", fontsize=14, fontweight="bold")
    plt.ylabel("Number of Detections")
    plt.xlabel("Acquisition Date")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Geospatial Hotspot Distribution (Latitude vs Longitude)

In [ ]:
if not df.empty and "latitude" in df.columns and "longitude" in df.columns:
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(
        df["longitude"], 
        df["latitude"], 
        c=df["frp"] if "frp" in df.columns else "#e67e22",
        cmap="YlOrRd", 
        alpha=0.6, 
        s=15,
        edgecolor="none"
    )
    if "frp" in df.columns:
        plt.colorbar(scatter, label="FRP (MW)")
    plt.title("Spatial Hotspot Distribution (Coordinates & Intensity)", fontsize=14, fontweight="bold")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.show()

## 7. FRP (Fire Radiative Power) Distribution

In [ ]:
frp_col = next((c for c in ["frp", "frp_mw"] if c in df.columns), None)
if not df.empty and frp_col:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram / KDE (log scale for highly skewed values)
    sns.histplot(df[frp_col], kde=True, ax=axes[0], color="#d35400", log_scale=True)
    axes[0].set_title(f"{frp_col.upper()} Distribution (Log Scale)", fontweight="bold")
    axes[0].set_xlabel("FRP (MW)")
    
    # Boxplot
    sns.boxplot(y=df[frp_col], ax=axes[1], color="#f39c12")
    axes[1].set_title(f"{frp_col.upper()} Boxplot", fontweight="bold")
    axes[1].set_ylabel("FRP (MW)")
    
    plt.tight_layout()
    plt.show()

## 8. Brightness Temperature Distribution

In [ ]:
bright_cols = [c for c in df.columns if "bright" in c]
if not df.empty and bright_cols:
    plt.figure(figsize=(12, 5))
    for col in bright_cols:
        sns.kdeplot(df[col].dropna(), label=col, fill=True, alpha=0.3)
    plt.title("Brightness Temperature Distributions (Kelvin)", fontsize=14, fontweight="bold")
    plt.xlabel("Temperature (K)")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 9. Detection Confidence Breakdown

In [ ]:
if not df.empty and "confidence" in df.columns:
    plt.figure(figsize=(8, 4))
    if pd.api.types.is_numeric_dtype(df["confidence"]):
        sns.histplot(df["confidence"], bins=20, kde=True, color="#27ae60")
        plt.title("MODIS Confidence Distribution (%)", fontsize=14, fontweight="bold")
        plt.xlabel("Confidence Score (0-100)")
    else:
        df["confidence"].value_counts().plot(kind="bar", color="#16a085", edgecolor="black")
        plt.title("VIIRS Confidence Category Frequencies", fontsize=14, fontweight="bold")
        plt.xlabel("Confidence Level")
        plt.ylabel("Count")
    plt.tight_layout()
    plt.show()